# 07 — Mechanism and Robustness of Δα̂

Why Δα̂ ≈ 0 (return gain vs. beta penalty), how it varies with cash, and whether it survives alternative screens,
outlier treatments and factor models. Every check uses the common fund set and the same-months rows from NB06.

| | |
|---|---|
| **Inputs** | `stage1b_corrected_returns.parquet` (NB05); `stage2a_common_results.parquet`, `stage2b_results.parquet`, `delta_alpha.parquet` (NB06) |
| **Outputs** | none (tables printed; Figure 2 saved) |
| **Thesis** | Section 5.3 (Tables 5–6, Figure 2), Sections 5.4.3–5.4.5 (Tables 7–8) |

**To replicate:** set `DATA_ROOT` in `thesis_config.py`, run NB01–NB06, then *Restart Kernel → Run All*.

## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

from thesis_config import (DATA_DIR, FACTORS, HAC_LAGS, MIN_OBS, assert_matches, check_paths, describe_cash, estimate_alpha,
                           max_consecutive_run, save_fig)

check_paths(*(DATA_DIR / f for f in ["stage1b_corrected_returns.parquet", "stage2a_common_results.parquet",
                                     "stage2b_results.parquet", "delta_alpha.parquet"]))

df         = pd.read_parquet(DATA_DIR / "stage1b_corrected_returns.parquet")
results_2a = pd.read_parquet(DATA_DIR / "stage2a_common_results.parquet")
results_2b = pd.read_parquet(DATA_DIR / "stage2b_results.parquet")
delta      = pd.read_parquet(DATA_DIR / "delta_alpha.parquet")
common_funds = sorted(delta["wficn"])

# Same-months estimation rows of the common funds (identical to NB06's regression samples)
frozen = (df[df["wficn"].isin(common_funds)]
          .dropna(subset=FACTORS + ["excess_ret", "excess_r_corrected"])
          .sort_values(["wficn", "caldt"], kind="mergesort"))
print(f"Estimation rows: {len(frozen):,} fund-months, {frozen['wficn'].nunique()} funds")
assert_matches("Section 5.4.5: estimation rows, funds", [len(frozen), frozen["wficn"].nunique()], [318_863, 1_950], 0)

def ann(s):
    """Monthly decimal -> %/yr."""
    return s * 1200

def summarise(tag, s):
    s = ann(s.dropna())
    t = stats.ttest_1samp(s, 0).statistic
    print(f"{tag:<26} n={len(s):>5}  mean={s.mean():+.4f}  median={s.median():+.4f}  t={t:+.2f}  share<0={(s < 0).mean():.3f}")
    return s.mean(), t

## 1. Mechanism: return gain vs. beta penalty (Section 5.3)

Because OLS is linear in the dependent variable, Δα̂<sub>i</sub> = mean(R̃ − R)<sub>i</sub> − Σ<sub>k</sub> Δβ̂<sub>i,k</sub>·f̄<sub>i,k</sub>
exactly, with f̄<sub>i,k</sub> the fund's own factor means over its estimation rows: a **return gain** minus a **beta penalty**.

In [ ]:
b = results_2b.merge(results_2a, on="wficn", suffixes=("_corr", ""))
BETAS = ["beta_mkt", "beta_smb", "beta_hml", "beta_rmw", "beta_cma", "beta_mom", "beta_liq"]

fund_fac_means = frozen.groupby("wficn")[FACTORS].mean().reindex(b["wficn"])
contrib = pd.DataFrame({f: (b[f"{bc}_corr"] - b[bc]).values * fund_fac_means[f].values
                        for f, bc in zip(FACTORS, BETAS)}, index=b["wficn"])     # per-fund penalty by factor

b["beta_penalty"] = contrib.sum(axis=1).values
b = b.merge(frozen.assign(g=frozen["excess_r_corrected"] - frozen["excess_ret"])
                  .groupby("wficn")["g"].mean().rename("ret_gain"), on="wficn")
b = b.merge(delta[["wficn", "delta_alpha"]], on="wficn")
b["predicted_dalpha"] = b["ret_gain"] - b["beta_penalty"]
assert np.allclose(b["predicted_dalpha"], b["delta_alpha"]), "decomposition identity fails"

gain, penalty, net = ann(b["ret_gain"].mean()), ann(b["beta_penalty"].mean()), ann(b["delta_alpha"].mean())
print(f"Return gain {gain:+.3f} − beta penalty {penalty:+.3f} = Δα̂ {net:+.3f}  (%/yr, mean over funds)")
print(f"Median: gain {ann(b['ret_gain'].median()):+.3f}, penalty {ann(b['beta_penalty'].median()):+.3f}")

# Effective cash weight and passive excess return behind the gain
er = frozen.assign(At=frozen["excess_r_corrected"] - frozen["excess_ret"], cash_dec=frozen["per_cash_clean"] / 100)
w_pool = er["cash_dec"].mean()
print(f"Effective cash weight w̄ (pooled fund-months): {w_pool:.2%} | "
      f"cash-weighted passive excess return: {ann(er['At'].mean() / w_pool):+.2f}%/yr")

assert_matches("Section 5.3: return gain, beta penalty, Δα̂ (%/yr)", [gain, penalty, net], [0.188, 0.205, -0.017], 3)
assert_matches("Section 5.3: cash weight w̄ (%)", w_pool * 100, 2.4, 1)


**Units and a worked example.** All three numbers are **percentage points per year** (monthly decimal returns × 1200),
averaged over funds. They are not percentages *of* a return: +0.188 means a fund earning 9.00% a year above the T-bill
rate earns 9.188% after the correction.

The beta penalty is **not** subtracted from the return. The corrected return really is 0.188 pp higher. The penalty is
the part of the fund's return that the factor model *explains* by its higher market beta, and alpha is what is left:
alpha = mean excess return − Σ<sub>k</sub> β<sub>k</sub>·f̄<sub>k</sub>. The table below shows this for the average fund.


In [ ]:

fund_means = frozen.groupby("wficn")[["excess_ret", "excess_r_corrected"]].mean().reindex(b["wficn"])
R_conv, R_corr = ann(fund_means["excess_ret"].mean()), ann(fund_means["excess_r_corrected"].mean())
a_conv, a_corr = ann(b["alpha"].mean()), ann(b["alpha_corr"].mean())

example = pd.DataFrame(
    {"Mean excess return": [R_conv, R_corr],
     "Explained by factors (Σβ·f̄)": [R_conv - a_conv, R_corr - a_corr],
     "Alpha": [a_conv, a_corr]},
    index=["Conventional (raw return)", "Corrected (cash → passive)"])
example.loc["Change"] = example.loc["Corrected (cash → passive)"] - example.loc["Conventional (raw return)"]
print("Average fund, percentage points per year:")
print(example.round(3).to_string())

# The change row IS the decomposition: return gain, beta penalty, Δα̂
assert np.allclose(example.loc["Change"], [gain, penalty, net])

print(f"""
Reading: the average fund earns {R_conv:.3f}% a year above the T-bill rate. Crediting its cash with the passive
benchmark raises this to {R_corr:.3f}% (+{gain:.3f} pp). Its median market beta rises from
{b['beta_mkt'].median():.3f} to {b['beta_mkt_corr'].median():.3f}, so the factor model now explains {penalty:.3f} pp more of
its return. That is slightly more than the gain, so alpha moves from {a_conv:.3f}% to {a_corr:.3f}% ({net:+.3f} pp).

Order of magnitude: {gain:.3f} pp/yr = {gain / 12 * 100:.2f} basis points per month (cf. the pooled 1.63 bps in Table 4);
for a fund with $1 billion under management, about ${gain / 100 * 1e3:.1f} million a year.""")



**Cash weight in the estimation sample.** `per_cash_clean` (% of TNA) on the 318,863 fund-months that enter the Stage 2
regressions — the weight the correction actually uses. The fund-level view is the input to Table 6.


In [ ]:

cash_est = pd.DataFrame({
    "Fund-months (estimation rows)": describe_cash(frozen["per_cash_clean"]),
    "Funds (time-series mean)":      describe_cash(frozen.groupby("wficn")["per_cash_clean"].mean()),
})
print(cash_est.round(2).to_string())


**Table 5 — per-factor contribution to the beta penalty** (mean over funds).

In [ ]:
m = ann(contrib.mean())
table5 = pd.DataFrame({"Contribution (%/yr)": m, "% of total penalty": 100 * m / m.sum()})
table5.loc["Total"] = [m.sum(), 100.0]
print(table5.round({"Contribution (%/yr)": 5, "% of total penalty": 1}).to_string())

THESIS_TABLE5 = [0.20990, 0.00990, -0.00040, -0.01900, -0.00090, 0.00120, 0.00450, 0.20516]
assert_matches("Table 5: contributions (%/yr)", table5["Contribution (%/yr)"], THESIS_TABLE5, 4)
assert_matches("Table 5: % of total penalty", table5["% of total penalty"],
               [102.3, 4.8, -0.2, -9.3, -0.5, 0.6, 2.2, 100.0], 1)

**Figure 2 — decomposition of the mean correction.**

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(["Return gain", "Beta penalty", "Net Δα̂"], [gain, penalty, net], bottom=[0, net, 0],
       color=["#1a9641", "#d7191c", "#2c7bb6"], width=0.6, edgecolor="none")
ax.plot([0.3, 0.7], [gain, gain], color="grey", lw=0.8, ls="--")
ax.plot([1.3, 1.7], [net, net],   color="grey", lw=0.8, ls="--")
ax.axhline(0, color="black", lw=1)
ax.text(0, gain + 0.008, f"{gain:+.3f}", ha="center", va="bottom", fontsize=11)
ax.text(1, (gain + net) / 2, f"{-penalty:+.3f}", ha="center", va="center", fontsize=11, color="white")
ax.text(2, net - 0.008, f"{net:+.3f}", ha="center", va="top", fontsize=11)
ax.set_ylabel("Percentage points per year", fontsize=12)
ax.set_title("Decomposition of Δα̂: return gain offset by beta penalty", fontsize=12)
ax.set_ylim(net - 0.06, gain + 0.06)
plt.tight_layout()
save_fig(fig, "fig2_mechanism_decomposition")
plt.show()

**Table 6 — Δα̂ by cash-weight quartile** (fund-level mean `per_cash_clean` over the whole panel).

In [ ]:
w_bar = df.groupby("wficn")["per_cash_clean"].mean().rename("w_bar")
q = delta.merge(w_bar, on="wficn", how="left").dropna(subset=["delta_alpha", "w_bar"])
q["quartile"] = pd.qcut(q["w_bar"], 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"])
g = q.groupby("quartile", observed=True)
table6 = pd.DataFrame({
    "n":                   g["delta_alpha"].size(),
    "mean cash (%)":       g["w_bar"].mean(),
    "Δα mean (%/yr)":      ann(g["delta_alpha"].mean()),
    "Δα median (%/yr)":    ann(g["delta_alpha"].median()),
    "conv. α mean (%/yr)": ann(g["alpha_conv"].mean()),
})
print(table6.round({"mean cash (%)": 2, "Δα mean (%/yr)": 3, "Δα median (%/yr)": 3, "conv. α mean (%/yr)": 3}).to_string())

THESIS_TABLE6 = [[488, 0.38, -0.026, -0.007, -1.070], [487, 1.52, -0.001, -0.004, -0.935],
                 [487, 2.40, -0.014, -0.004, -0.980], [488, 5.13, -0.028, -0.001, -0.764]]
T6 = np.array(THESIS_TABLE6)
assert_matches("Table 6: n", table6["n"], T6[:, 0], 0)
assert_matches("Table 6: mean cash (%)", table6["mean cash (%)"], T6[:, 1], 2)
assert_matches("Table 6: Δα mean, median, conv. α (%/yr)",
               table6[["Δα mean (%/yr)", "Δα median (%/yr)", "conv. α mean (%/yr)"]], T6[:, 2:], 3)

## 2. Section 5.4.3 — minimum-history screen (Table 7)

Tightening the consecutive-history requirement only removes funds, so Δα̂ is the mean over the surviving subset
(no re-estimation). The longest consecutive run is measured on each fund's months with a raw return and all factors.

In [ ]:
consec = (df.dropna(subset=FACTORS + ["excess_ret"]).groupby("wficn")["caldt"]
            .apply(lambda s: max_consecutive_run(list(s))).rename("consec_run"))
dd = delta.merge(consec, on="wficn")

table7 = {}
for k in [60, 72, 84, 96, 120]:
    s = dd.loc[dd["consec_run"] >= k, "delta_alpha"]
    table7[k] = (len(s),) + tuple(summarise(f">= {k} months", s))

short = dd.loc[dd["consec_run"] < 60, ["wficn", "consec_run"]]
print(f"\nCommon-set funds whose longest consecutive run is below 60 months: {len(short)}")
print(short.to_string(index=False))
assert_matches("Table 7: funds below 60 consecutive months", len(short), 1, 0)

THESIS_TABLE7 = {60: (1949, -0.0173, -2.76), 72: (1811, -0.0161, -2.50), 84: (1701, -0.0146, -2.29),
                 96: (1586, -0.0133, -1.99), 120: (1285, -0.0182, -2.72)}
T7, E7 = np.array(list(table7.values())), np.array(list(THESIS_TABLE7.values()))
assert_matches("Table 7: n", T7[:, 0], E7[:, 0], 0)
assert_matches("Table 7: Δα̂ mean (%/yr)", T7[:, 1], E7[:, 1], 4)
assert_matches("Table 7: t", T7[:, 2], E7[:, 2], 2)

*Note.* The 60-month row has 1,949 funds, not 1,950: the fund(s) listed above satisfied the 60-consecutive-month
screen in NB01 before step 9 (pre-coverage index-month removal) shortened their history. They remain in the common
set because NB06 requires 60 *total* same-months observations, not 60 consecutive ones.

## 3. Section 5.4.4 — winsorisation

1. **Dependent variable:** both legs re-estimated with the return winsorised at its pooled 1st/99th percentile
   (same funds, same rows).
2. **Cross-section:** the 1,950 per-fund Δα̂ are clipped at their own 1st/99th percentile; betas untouched.

In [ ]:
res_2b_w, _ = estimate_alpha(df, common_funds, "excess_r_corrected", "2b winsorised", winsorize=True)
res_2a_w, _ = estimate_alpha(df, common_funds, "excess_ret",         "2a winsorised", winsorize=True)
delta_w = (res_2a_w[["wficn", "alpha"]].rename(columns={"alpha": "alpha_conv_w"})
           .merge(res_2b_w[["wficn", "alpha"]].rename(columns={"alpha": "alpha_corr_w"}), on="wficn"))
delta_w["delta_alpha_w"] = delta_w["alpha_corr_w"] - delta_w["alpha_conv_w"]

d = delta["delta_alpha"].dropna()
lo, hi = d.quantile(0.01), d.quantile(0.99)
print()
r_primary = summarise("primary",                 d)
r_xsec    = summarise("cross-section winsorised", d.clip(lo, hi))
r_dv      = summarise("return winsorised",        delta_w["delta_alpha_w"])

assert_matches("Section 5.4.4: means (%/yr) primary / cross-section / return-winsorised",
               [r_primary[0], r_xsec[0], r_dv[0]], [-0.0173, -0.015, 0.008], 3)
assert_matches("Section 5.4.4: t-statistics", [r_primary[1], r_xsec[1], r_dv[1]], [-2.76, -2.96, 1.66], 2)

## 4. Section 5.4.5 — alternative factor models (Table 8)

FF3 and Carhart-4 against the primary seven-factor model, on the identical estimation rows; only the regressor set changes.

In [ ]:
SPECS = {"FF3": ["mkt_rf", "smb", "hml"], "Carhart-4": ["mkt_rf", "smb", "hml", "mom"], "7F (primary)": FACTORS}

rows = []
for fund, f in frozen.groupby("wficn", sort=False):
    if len(f) < MIN_OBS:
        continue
    for spec, cols in SPECS.items():
        Xc = add_constant(f[cols].values)
        p_conv = OLS(f["excess_ret"].values,         Xc).fit(cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS}).params
        p_corr = OLS(f["excess_r_corrected"].values, Xc).fit(cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS}).params
        rows.append({"wficn": fund, "spec": spec, "delta_alpha": p_corr[0] - p_conv[0],
                     "dbeta_mkt": p_corr[1] - p_conv[1]})
fm = pd.DataFrame(rows)

g = fm.groupby("spec")
table8 = pd.DataFrame({
    "n":                g["delta_alpha"].size(),
    "Δα mean (%/yr)":   ann(g["delta_alpha"].mean()),
    "Δα median (%/yr)": ann(g["delta_alpha"].median()),
    "t":                g["delta_alpha"].apply(lambda s: stats.ttest_1samp(s, 0).statistic),
    "share < 0":        g["delta_alpha"].apply(lambda s: (s < 0).mean()),
    "median Δβ_mkt":    g["dbeta_mkt"].median(),
    "mean Δβ_mkt":      g["dbeta_mkt"].mean(),
}).reindex(SPECS)
rounding = {"Δα mean (%/yr)": 4, "Δα median (%/yr)": 4, "t": 2, "share < 0": 3, "median Δβ_mkt": 4, "mean Δβ_mkt": 4}
print(table8.round(rounding).to_string())

quant = fm.assign(d=ann(fm["delta_alpha"])).groupby("spec")["d"].quantile([.05, .10, .25, .50, .75, .90, .95]).unstack()
quant.columns = [f"p{int(c*100)}" for c in quant.columns]
print("\nPer-fund Δα̂ quantiles (%/yr):")
print(quant.reindex(SPECS).round(4).to_string())

THESIS_TABLE8 = [[1950, -0.0417, -0.0189, -6.02, 0.575, 0.0186],
                 [1950, -0.0394, -0.0185, -5.73, 0.572, 0.0182],
                 [1950, -0.0173, -0.0050, -2.76, 0.524, 0.0179]]
T8 = np.array(THESIS_TABLE8)
assert_matches("Table 8: n", table8["n"], T8[:, 0], 0)
assert_matches("Table 8: Δα mean, median (%/yr), median Δβ_mkt",
               table8[["Δα mean (%/yr)", "Δα median (%/yr)", "median Δβ_mkt"]], T8[:, [1, 2, 5]], 4)
assert_matches("Table 8: t", table8["t"], T8[:, 3], 2)
# Section 5.4.5 text lists the mean Δβ_mkt as "+0.023, +0.023 and +0.024" (FF3, Carhart-4, 7F); the order is reversed
assert_matches("Section 5.4.5: mean Δβ_mkt FF3 / Carhart-4 / 7F", table8["mean Δβ_mkt"], [0.024, 0.023, 0.023], 3)
assert_matches("Table 8: share < 0", table8["share < 0"], T8[:, 4], 3)
dispersion = (quant["p95"] - quant["p5"]).reindex(SPECS)
print("\np95 − p5 dispersion (%/yr): " + ", ".join(f"{s} {v:.2f}" for s, v in dispersion.items()))
assert_matches("Section 5.4.5: p95 − p5 dispersion FF3 / Carhart-4 / 7F (%/yr)", dispersion, [0.70, 0.69, 0.67], 2)